# 03 — State-Level EDA & Causal Inference Panel

State-level exploratory analysis comparing organic waste ban states vs. non-ban states.

**Outputs:** 4 charts in `outputs/causal/charts/`
1. State ranking by landfill rate
2. State ranking by diversion rate
3. Ban vs non-ban box plots
4. Landfill rate over time (ban vs non-ban)

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('outputs/causal/charts', exist_ok=True)

BAN_COLOR, NOBAN_COLOR = '#2C5F2D', '#888888'

---
## Build State-Year Panel

In [2]:
from utils.causal_inference import build_state_year_panel

state_year, policy = build_state_year_panel()
print(f'Panel shape: {state_year.shape}')
print(f'Policy timeline: {policy.shape}')
print(f'Ban states: {state_year[state_year["is_ban_state"]]["state"].nunique()}')

sy2024 = state_year[state_year['year'] == 2024]

Panel shape: (750, 18)
Policy timeline: (12, 4)
Ban states: 12


---
## Chart 1: State Ranking by Landfill Rate

In [3]:
fig, ax = plt.subplots(figsize=(10, 14))
sy = sy2024.sort_values('landfill_rate')
colors = [BAN_COLOR if b else NOBAN_COLOR for b in sy['is_ban_state']]
ax.barh(sy['state'], sy['landfill_rate'], color=colors)
ax.set_xlabel('Landfill Rate (2024)')
ax.set_title('State Rankings by Landfill Rate (2024)\nGreen = Ban State, Gray = No Ban')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/state_ranking_landfill.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved state_ranking_landfill.png')

Saved state_ranking_landfill.png


### Chart 2: State Ranking by Diversion Rate

In [4]:
fig, ax = plt.subplots(figsize=(10, 14))
sy = sy2024.sort_values('diversion_rate')
colors = [BAN_COLOR if b else NOBAN_COLOR for b in sy['is_ban_state']]
ax.barh(sy['state'], sy['diversion_rate'], color=colors)
ax.set_xlabel('Diversion Rate (2024)')
ax.set_title('State Rankings by Diversion Rate (2024)\nGreen = Ban State, Gray = No Ban')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/state_ranking_diversion.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved state_ranking_diversion.png')

Saved state_ranking_diversion.png


### Chart 3: Ban vs Non-Ban Comparison (Box Plots)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for i, (m, l) in enumerate([('landfill_rate', 'Landfill Rate'), ('diversion_rate', 'Diversion Rate')]):
    ban = sy2024[sy2024['is_ban_state']][m]
    noban = sy2024[~sy2024['is_ban_state']][m]
    bp = axes[i].boxplot([ban, noban], labels=['Ban (12)', 'Non-Ban (38)'], patch_artist=True, widths=0.6)
    bp['boxes'][0].set_facecolor(BAN_COLOR)
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(NOBAN_COLOR)
    bp['boxes'][1].set_alpha(0.7)
    axes[i].set_ylabel(l)
    axes[i].set_title(f'{l}: Ban vs Non-Ban (2024)')
    axes[i].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/state_ban_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved state_ban_comparison.png')

Saved state_ban_comparison.png


### Chart 4: Landfill Rate Over Time — Ban vs Non-Ban

In [6]:
ban_trend = state_year[state_year['is_ban_state']].groupby('year')['landfill_rate'].mean()
noban_trend = state_year[~state_year['is_ban_state']].groupby('year')['landfill_rate'].mean()
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ban_trend.index, ban_trend.values, 'o-', color=BAN_COLOR, lw=2, label='Ban States')
ax.plot(noban_trend.index, noban_trend.values, 's-', color=NOBAN_COLOR, lw=2, label='Non-Ban States')
ax.axvline(x=2014, color='red', ls='--', alpha=0.5, label='First bans (2014)')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Landfill Rate')
ax.set_title('Landfill Rate Over Time: Ban vs Non-Ban States')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/state_ban_comparison_over_time.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved state_ban_comparison_over_time.png')

Saved state_ban_comparison_over_time.png


---
## Summary

All 4 state EDA charts saved to `outputs/causal/charts/`:
1. **state_ranking_landfill.png** — All 50 states ranked by landfill rate, ban states highlighted
2. **state_ranking_diversion.png** — All 50 states ranked by diversion rate, ban states highlighted
3. **state_ban_comparison.png** — Box plots comparing ban vs non-ban states on landfill and diversion rates
4. **state_ban_comparison_over_time.png** — Temporal trend showing divergence after first bans in 2014